# Encrypted Math Tutorial: Basic Operations (`math/basic.py`)

This tutorial covers `src/concrete_fhe_toolkit/math/basic.py`, which provides fundamental mathematical and logical operations for encrypted integers. FHE requires all values and intermediate results to be bounded, so these functions often require `min_value` and `max_value` to construct lookup tables.

## 1. Basic Arithmetic (`add`, `subtract`, `multiply`, `square`, `cube`, `negate`)
While addition and subtraction are somewhat native to FHE, multiplication consumes the multiplicative depth of the circuit. Squaring and cubing are optimized via univariate lookup tables.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.math.basic import add, subtract, multiply, square, cube, negate

def test_arithmetic(a: int, b: int):
    return add(a, b), subtract(a, b), multiply(a, b), square(a), cube(a), negate(a)

# 1. Cleartext
res_add, res_sub, res_mul, res_sq, res_cb, res_neg = test_arithmetic(4, 3)
assert res_add == 7
assert res_sub == 1
assert res_mul == 12
assert res_sq == 16
assert res_cb == 64
assert res_neg == -4
print("Cleartext arithmetic passed!")

# 2. FHE Compilation
compiler = fhe.Compiler(test_arithmetic, {"a": "encrypted", "b": "encrypted"})
inputset = [(0, 0), (4, 3), (1, 5)]
circuit = compiler.compile(inputset)

# 3. Encrypted Execution
enc_res = circuit.encrypt_run_decrypt(4, 3)
assert enc_res == (7, 1, 12, 16, 64, -4)
print("✅ Encrypted arithmetic passed!")

## 2. Logical Predicates (`equal`, `less`, `greater`, `is_zero`)
These functions perform encrypted comparisons. Because dynamic branching (`if a > b:`) is impossible, these return exactly `1` or `0` so they can be mathematically multiplied as masks.

In [ ]:
from concrete_fhe_toolkit.math.basic import equal, not_equal, less, greater, is_zero

def test_logic(a: int, b: int):
    return equal(a, b), not_equal(a, b), less(a, b), greater(a, b), is_zero(a)

res_eq, res_neq, res_lt, res_gt, res_z = test_logic(5, 8)
assert res_eq == 0
assert res_neq == 1
assert res_lt == 1
assert res_gt == 0
assert res_z == 0
print("Cleartext logic passed!")

compiler = fhe.Compiler(test_logic, {"a": "encrypted", "b": "encrypted"})
inputset = [(0, 0), (5, 8), (10, 2)]
circuit = compiler.compile(inputset)

enc_res = circuit.encrypt_run_decrypt(5, 8)
assert enc_res == (0, 1, 1, 0, 0)
print("✅ Encrypted logic passed!")

## 3. Clamping and Absolute Values (`make_clamp`, `make_absolute`)
Bounding values within a specific range is one of the most critical steps in deep FHE circuits to prevent the bit-width from exploding.

In [ ]:
from concrete_fhe_toolkit.math.basic import make_clamp, make_absolute

# Create clamp function: input range [-20, 20], clamp to [-5, 5]
clamp_fn = make_clamp(min_input=-20, max_input=20, min_value=-5, max_value=5)
abs_fn = make_absolute(min_value=-20, max_value=20)

def test_bounds(a: int):
    return clamp_fn(a), abs_fn(a)

assert test_bounds(-15) == (-5, 15)  # Clamped to -5, Abs is 15
assert test_bounds(3) == (3, 3)      # Unchanged, Abs is 3
assert test_bounds(20) == (5, 20)    # Clamped to 5, Abs is 20
print("Cleartext bounds passed!")

compiler = fhe.Compiler(test_bounds, {"a": "encrypted"})
inputset = [(-20,), (0,), (20,)]
circuit = compiler.compile(inputset)

enc_res1 = circuit.encrypt_run_decrypt(-15)
enc_res2 = circuit.encrypt_run_decrypt(20)
assert enc_res1 == (-5, 15)
assert enc_res2 == (5, 20)
print("✅ Encrypted bounds passed!")

## 4. Modulo and DivMod (`make_modulo`, `make_divmod`)
Like division, modulo operations are extremely expensive without dynamic branching. We use explicit zero-handling parameters to secure the circuits against division by zero.

In [ ]:
from concrete_fhe_toolkit.math.basic import make_modulo, make_divmod

mod_fn = make_modulo(min_numerator=0, max_numerator=20, min_denominator=0, max_denominator=10, zero_result=99)
divmod_fn = make_divmod(min_numerator=0, max_numerator=20, min_denominator=0, max_denominator=10, zero_quotient=88, zero_remainder=99)

def test_modulo(num: int, den: int):
    return mod_fn(num, den), divmod_fn(num, den)

assert test_modulo(17, 5) == (2, (3, 2))  # 17 % 5 = 2. DivMod = (3, 2)
assert test_modulo(17, 0) == (99, (88, 99)) # Zero handling!
print("Cleartext modulo passed!")

compiler = fhe.Compiler(test_modulo, {"num": "encrypted", "den": "encrypted"})
inputset = [(0, 1), (17, 5), (17, 0)]
circuit = compiler.compile(inputset)

enc_res1 = circuit.encrypt_run_decrypt(17, 5)
enc_res2 = circuit.encrypt_run_decrypt(17, 0)
assert enc_res1 == (2, (3, 2))
assert enc_res2 == (99, (88, 99))
print("✅ Encrypted modulo passed all zero-division edge cases!")